# 02 — Validate and visualize nucleus instance masks

This notebook performs detailed quality control of the nucleus labels created in notebook 01. The filename replaces the old Cellpose-specific notebook because the current default detector runs internally; external Cellpose labels remain compatible.

The manifest column is still named `cellpose_mask_path` for backward compatibility, but it means **the active nucleus instance-label path**, regardless of which detector produced it.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

## User settings

Set `IMAGE_ID=None` to inspect the first manifest row. The optional parameter experiment does not replace stored labels unless you later rerun `prepare_dataset.py` with chosen values.

In [ ]:
MANIFEST_PATH = Path('data/metadata/manifest.csv')
IMAGE_ID = None
MAX_NUCLEUS_DISTANCE = 64.0
RUN_PARAMETER_EXPERIMENT = False

# Experimental values used only when RUN_PARAMETER_EXPERIMENT=True.
EXPERIMENT_GAUSSIAN_SIGMA = 1.2
EXPERIMENT_THRESHOLD_SCALE = 1.0
EXPERIMENT_MIN_NUCLEUS_AREA = 30
EXPERIMENT_MIN_PEAK_DISTANCE = 7

## Load the aligned microscopy image and nucleus labels

Instance labels must be a two-dimensional numeric array with exactly the same height and width as the microscopy image. `0` is background and each positive integer identifies one nucleus.

In [ ]:
import numpy as np
import pandas as pd
import tifffile

from astroseg.io import get_channel, load_manifest, load_ome_tiff
from astroseg.preprocessing import (
    create_nucleus_proximity_map,
    detect_nucleus_instances,
    labels_to_binary_mask,
    percentile_normalize,
    validate_nucleus_labels,
)

manifest = load_manifest(MANIFEST_PATH)
selected_id = IMAGE_ID or str(manifest.iloc[0]['image_id'])
matches = manifest.loc[manifest['image_id'] == selected_id]
if len(matches) != 1:
    raise ValueError(f'IMAGE_ID must identify one manifest row; found {len(matches)}')
row = matches.iloc[0]

def resolve_manifest_path(value, manifest_path=MANIFEST_PATH):
    path = Path(str(value))
    for candidate in (path, manifest_path.parent / path):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(value)

if not str(row['cellpose_mask_path']).strip():
    raise ValueError('Nucleus label path is empty. Complete notebook 01 first.')
microscopy = load_ome_tiff(resolve_manifest_path(row['path']))
label_path = resolve_manifest_path(row['cellpose_mask_path'])
labels = np.load(label_path, allow_pickle=False) if label_path.suffix.lower() == '.npy' else tifffile.imread(label_path)
labels = np.asarray(labels)
validate_nucleus_labels(labels, microscopy.image.shape[-2:])
binary = labels_to_binary_mask(labels)
proximity = create_nucleus_proximity_map(binary, MAX_NUCLEUS_DISTANCE)
gfap = get_channel(microscopy, str(row['gfap_channel']))
dapi = get_channel(microscopy, str(row['dapi_channel']))
print('Image:', selected_id)
print('Labels:', label_path)
print('Shape / dtype:', labels.shape, '/', labels.dtype)
print('Nucleus instances:', int(labels.max(initial=0)))
print('Nucleus foreground fraction:', float(binary.mean()))

## Visual alignment check

The DAPI panel should align with the colored instances. The binary mask shows the exact context plane given to the astrocyte model; the proximity map is the third model input channel.

In [ ]:
import matplotlib.pyplot as plt

panels = [
    ('GFAP', percentile_normalize(gfap), 'gray'),
    ('DAPI', percentile_normalize(dapi), 'gray'),
    ('Nucleus instances', labels, 'nipy_spectral'),
    ('Binary nucleus mask', binary, 'gray'),
    ('Nucleus proximity', proximity, 'magma'),
]
figure, axes = plt.subplots(2, 3, figsize=(16, 10), squeeze=False)
for axis, (title, image, cmap) in zip(axes.flat, panels):
    axis.imshow(image, cmap=cmap)
    axis.set_title(title)
    axis.axis('off')
for axis in axes.flat[len(panels):]:
    axis.axis('off')
figure.suptitle(selected_id)
figure.tight_layout()
plt.show()

In [ ]:
from skimage.segmentation import find_boundaries

base = np.repeat(percentile_normalize(dapi)[..., None], 3, axis=2)
boundaries = find_boundaries(labels, mode='outer')
overlay = base.copy()
overlay[boundaries] = np.array([1.0, 0.1, 0.1])
plt.figure(figsize=(9, 9))
plt.imshow(overlay)
plt.title('Nucleus boundaries over DAPI')
plt.axis('off')
plt.show()

## Instance-size diagnostics

Very large objects may indicate merged nuclei; many very small objects may indicate noise or over-segmentation. Pixel areas are useful for QC but are not physical areas unless pixel-size metadata is available and applied.

In [ ]:
areas = np.bincount(labels.astype(np.int64, copy=False).ravel())[1:]
areas = areas[areas > 0]
area_summary = pd.Series(areas, name='area_pixels').describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
area_summary

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(areas, bins=50)
plt.xlabel('Instance area (pixels)')
plt.ylabel('Count')
plt.title(f'Nucleus area distribution: {selected_id}')
plt.show()

## Optional detector-parameter experiment

The internal detector is classical, not a neural network. `threshold_scale` changes foreground sensitivity; `min_peak_distance` controls how closely watershed markers may occur. Compare visually before changing defaults across a dataset. This cell does not overwrite any file.

In [ ]:
if RUN_PARAMETER_EXPERIMENT:
    experiment = detect_nucleus_instances(
        dapi,
        gaussian_sigma=EXPERIMENT_GAUSSIAN_SIGMA,
        threshold_scale=EXPERIMENT_THRESHOLD_SCALE,
        min_nucleus_area=EXPERIMENT_MIN_NUCLEUS_AREA,
        min_peak_distance=EXPERIMENT_MIN_PEAK_DISTANCE,
    )
    figure, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(labels, cmap='nipy_spectral')
    axes[0].set_title(f'Stored labels: {int(labels.max())} instances')
    axes[1].imshow(experiment.labels, cmap='nipy_spectral')
    axes[1].set_title(f'Experiment: {experiment.instance_count} instances')
    for axis in axes:
        axis.axis('off')
    plt.show()
else:
    print('Parameter experiment skipped. Set RUN_PARAMETER_EXPERIMENT=True to enable it.')

## Completion checklist

Accept the nucleus baseline when most DAPI-positive nuclei are represented, touching nuclei are reasonably separated, labels are aligned, and obvious noise is limited. Record parameter changes rather than tuning every image independently.

Continue with `03_create_initial_annotations.ipynb` to generate the automatic GFAP proposal.